## Problem 1 - Discrete Mathematics: Shareholding Networks *(compulsory, 25 points)*
A business-information company stores shareholding data for $n$ legal entities as a directed graph $G = (V,E)$, where an edge $(u, v)$ means company $u$ directly holds shares in company $v$.

1. **(27%)** Define "circular shareholding" in graph-theoretic terms. Then prove that $G$ has no circular shareholding if and only if the companies can be ordered so that every edge points from an earlier company to a later one (a topological ordering). Prove both directions.

Circular shareholding คือรูปแบบการถือหุ้นไขว้ระหว่างบริษัทตั้งแต่ 3 บริษัทขึ้นไปเป็นวงกลม\
ในเชิงทฤษฎีกราฟ ถ้าเกิดแต่ละบริษัทเป็น node และ การถือหุ้นจะทำให้เกิด edge ที่แสดงการถือหุ้น เช่นถ้า บริษัท A ถือหุ้น บริษัท B จะทำให้เกิด A -> B ซึ่ง circular shareholding จะเกิดขึ้นก็ต่อเมื่อมีการถือหุ้นกันเป็นวงกลม (เมื่อพยายามเดินไปตามทาง edge จะมีทิศทางที่กลับมาจุดเริ่มต้นได้) เช่น เมื่อ A -> B, B -> C และ C -> A

***พิสูจน์***\
**ถ้า G ไม่มี circular shareholding แล้ว บริษัทจะสามารถเรียงเป็นเส้นตรง และ edge ชี้ไปทางเดียวกันได้** 
เมื่อไม่มี circular shareholding หมายถึงว่าต้องมีอย่างน้อย 1 บริษัทที่ไม่มีบริษัทอื่นถือหุ้นอยู่เสมอ ดังนั้น เมื่อลองทำการตัดบริษัทที่ไม่มีบริษัทอื่นถือหุ้นอยู่ออกไป คุณสมบัติเดิมของการไม่มี circular shareholding จะยังคงอยู่เสมอ

**ถ้า บริษัทจะสามารถเรียงเป็นเส้นตรง และ edge ชี้ไปทางเดียวกันได้ แล้ว G ไม่มี circular shareholding** 
การที่สามารถเรียงไปทางเดียวกันได้โดยไม่มี edge เส้นไหนชี้ย้อนกลับ แสดงให้เห็นว่าไม่มีบริษัทไหนที่มีการถือหุ้นย้อนกลับมาที่บริษัทก่อนหน้าได้ แสดงให้เห็นว่า circular shareholding จะไม่สามารถเกิดขึ้นได้

2. **(33%)** Design an algorithm that returns every group of companies involved in circular shareholding with one another. State its time complexity in Big-O of $|V|$ and $|E|$ with justification. Implement it in Python and test it on a random graph with 100,000 nodes. Do not use ready-made functions from graph libraries (you may use them to verify your answer).

Algorithm ใช้หลักการที่ว่า ถ้าเป็น circular ไม่ว่าจะเดินไปตามกราฟหรือย้อนศรกราฟจะสามารถเดินวนได้เหมือนกัน\
ดังนั้นในขั้นแรกจะทำการสร้าง directed กราฟของแต่ละ node และ กราฟที่เกิดจากการ reverse edge\
จากนั้นทางการ DFS เพื่อหาลำดับของการเจอในกราฟ โดย node ที่ขึ้นมาก่อนบน list คืออยู่ปลายกราฟ และ node ที่อยู่สุดท้ายคือ node ที่มีแนวโน้ม เป็น root ที่สุด\
หลักจากนั้นจึงลองทำการตรวจสอบกราฟที่เป็น reverse โดยเริ่มจาก node สุดท้ายที่มีแนวโน้มเป็น root ที่สุด เพื่อตรวจสอบว่าสามารถเดินทางได้ไหม ถ้าได้แสดงว่าเกิด circular ขึ้น ถ้าเกิด dead end แสดงว่า node นั้นไม่ได้เป็น circular\
(ได้มีการตรวจสอบ algorithm ที่ src/q1/test_shareholding.py)

In [1]:
def circular_shareholding(n, edges):
  # Build the graph and the reversed graph
  adj = [[] for _ in range(n)]
  rev_adj = [[] for _ in range(n)]
    
  for u, v in edges:
    adj[u].append(v)
    rev_adj[v].append(u)
        
  # DFS to find last order traversal
  visited = [False] * n
  finished = [False] * n
  order = []
    
  for i in range(n):
    if not visited[i]:
      stack = [i]
      while stack:
        curr = stack[-1]
        if not visited[curr]:
          visited[curr] = True
          # Push unvisited neighbors to stack
          for neighbor in adj[curr]:
            if not visited[neighbor]:
              stack.append(neighbor)
        else:
          curr = stack.pop()
          # Only append to order when finished with all children
          if not finished[curr]:
            finished[curr] = True
            order.append(curr)
                        
  # Find circular groups using the reversed graph
  visited_rev = [False] * n
  circular_groups = []

  # Process in reverse of finishing times
  for i in reversed(order):
    if not visited_rev[i]:
      stack = [i]
      visited_rev[i] = True
      current_scc = []
      
      while stack:
        curr = stack.pop()
        current_scc.append(curr)
        
        for neighbor in rev_adj[curr]:
          if not visited_rev[neighbor]:
            visited_rev[neighbor] = True
            stack.append(neighbor)
        
        # > 1 companies in the SCC
        if len(current_scc) > 1:
          if current_scc not in circular_groups:
            circular_groups.append(current_scc)

  return circular_groups

In [3]:
import random

V = 100000
E = 250000

print("Generating graph")
edges = [(random.randint(0, V - 1), random.randint(0, V - 1)) for _ in range(E)]

circular_groups = circular_shareholding(V, edges)
print(f"New algorithm")
print(f"Found {len(circular_groups)} distinct circular shareholding groups.")
print(circular_groups)


Generating graph
New algorithm
Found 1 distinct circular shareholding groups.
[[1, 88542, 72080, 78194, 36, 42508, 57010, 26640, 51973, 91287, 93914, 20458, 11800, 71076, 91422, 72842, 84108, 40416, 33826, 24560, 7499, 68632, 18681, 10975, 24442, 49020, 7612, 48174, 17576, 54743, 77687, 26354, 25588, 1367, 1892, 68536, 78770, 58817, 16781, 7606, 48860, 27861, 40568, 31467, 90429, 55082, 39511, 10524, 5088, 60898, 33365, 98500, 33238, 10875, 5691, 70356, 28342, 68505, 86864, 18092, 80221, 69292, 88721, 11244, 98395, 92923, 29685, 13832, 16298, 54076, 28979, 49989, 28389, 81022, 63406, 74115, 32617, 71704, 188, 33598, 47895, 17357, 64325, 7210, 21344, 79981, 17453, 78283, 60708, 23747, 11377, 28812, 95912, 12407, 88043, 77921, 35577, 11313, 58962, 5485, 64104, 82313, 6259, 79837, 83745, 17923, 86610, 30878, 37507, 53124, 54671, 37751, 6383, 85517, 2177, 54887, 10327, 88923, 77861, 54933, 50880, 10728, 72178, 80905, 582, 262, 69628, 42573, 7869, 46254, 34057, 94595, 53693, 11159, 23978, 2

3. **(40%)** A registry holds $n=2,000,000$ company records, and you must run entity resolution to find duplicates.

a. If every pair is compared, how many comparisons are needed?

**ประมาณ 1,999,999,000,000 ครั้ง**\
การเปรียบเทียบสามารถใช้สูตรการจัดกลุ่มทีละ 2 ได้ (คำนวณที่ src/q1/combination.py)\
สูตรคำนวณคือ $$TotalComparisons = \frac{n*(n - 1)}{2}$$ 

b. If records are split into $k$ equal-sized blocks (blocking) and compared only within blocks, how many comparisons are needed? Give a formula in $n$ and $k$, then evaluate it for $k=1,000$.

**ประมาณ 1,999,000,000 ครั้ง**\
โดยใน 1 กลุ่ม จะมี 2,000 บริษัท ทำให้เกิดการจับคู่ในกลุ่มทั้งหมด 1,999,000 ครั้ง ทำให้ทั้งหมดต้องจับคู่ 1,999,000,000 ครั้ง (คำนวณที่ src/q1/combination.py)\
สูตรคำนวณโดยคือ $$TotalComparisons = k*\frac{n/k*(n/k - 1)}{2} = \frac{n*(n/k - 1)}{2}$$ 

c. Prove that when block sizes are unequal, the number of comparisons is always greater than or equal to the equal-size case (hint: convexity).

ลักษณะของกราฟ จากสูตร $f(x) = \frac{x*(x - 1)}{2}$ ในข้อ 3a เป็นลักษณะโค้งหงายขึ้น ซึ่งรู้ได้จากการที่ $$f''(x) = 1$$ ซึ่งมากกว่า 0 และจาก Jensen's Inequality ได้มีการบอกไว้ว่า $$f(\sum_{i=1}^{n}w_{i}x_{i}) ≤ \sum_{i=1}^{n}{w_{i}f(x_{i})}$$ ซึ่งหมายความว่า ค่าของฟังก์ชันที่ได้จากค่าเฉลี่ยถ่วงน้ำหนักของจุดต่าง ๆ จะน้อยกว่าหรือเท่ากับ ค่าเฉลี่ยถ่วงน้ำหนักของค่าฟังก์ชันที่นั้น ๆ เสมอ
ซึ่งหมายความว่าผลรวมของการที่ block เท่ากัน จะมากกว่าหรือเท่ากับการที่จัด block ที่ไม่เท่ากันเสมอ

เช่นในการแบ่งกลุ่ม 2 กลุ่ม\
ถ้าแบ่ง 1,000,000 2 กลุ่มเท่ากัน จะมีการเปรียบเทียบ  999,999,000,000 ครั้ง\
ถ้าแบ่งเป็น 2 กลุ่มที่ไม่เท่ากัน ที่ 900,000 และ 1,100,000 จะมีการเปรียบเทียบ 1,009,999,000,000 ครั้ง\
ซึ่งมีจำนวนมากกว่า (คำนวณที่ src/q1/combination.py)

d. Explain which kinds of duplicate pairs blocking can miss, and how you would mitigate this.

จะทำให้หาค่าซ้ำใน กรณีที่ค่าที่ซ้ำกับอยู่กันคนละ block ไม่เจอ เนื่องจากเป็นแค่การเปรียบเทียบแค่ภายใน block 
โดยเราจะสามารถแก้ปัญหานี้ได้ 2 วิธี
1. ทำการแบ่งว่าแต่ละ block เป็น block สำหรับอะไร ดังนั้นค่าประเภทเดียวกันก็จะอยู่ใน block เดียวกัน ซึ่งหมายถึงค่าที่เหมือนกันจะจะอยู่ใน block เดียวกันอย่างแน่นอน
2. ทำการ sort ข้อมูลตั้งแต่ก่อนใส่ค่าเพื่อให้ข้อมูลทุกอย่างเรียงต่อกัน และในการเปรียบเทียบให้เพื่อการเปรียบเทียบในค่าสุดท้ายของ block กับค่าแรกใน block ถัดไปเพื่อตรวจสอบว่าค่านั้นมีหรือยัง

ซึ่งทั้ง 2 วิธีมีข้อดีข้อเสียต่างกันข้อ 1 จำเป็นต้องการเกณฑ์ในการแบ่งแยกแต่งละ block และแต่ละ block จะมีความเป็นไปได้สูงที่จะมีขนาดที่ไม่เท่ากัน ซึ่งหมายถึงการเปรียบเทียบจำนวณครั้งก็จะมากขึ้นตามที่ได้มีการพิสูจน์ไปในข้อ 3c
ส่วนวิธีที่ 2 เมื่อมีการเพิ่มเติมข้อมูลมาใหม่ต้องมีการตรวจสอบข้อมูลกับ block ต่าง ๆ อยู่เสมอ และควรจะต้องทำการจัดระเบียบให้ดี เพื่อป้องกันข้อผิดพลาดที่จะใส่ข้อมูลผิด block ซึ่งจะทำให้เกิดปัญหาในระยะยาวเมื่อมีการใส่ข้อมูลผิดพลาด หรือรวมถึงอาจจะต้องมีการย้ายข้อมูลระหว่าง block ในกรณีที่ต้องการให้ข้อมูลแต่ละ block มีขนาดเท่ากัน

## Problem 2 - Linear Algebra: Indirect Ownership and PCA *(compulsory, 25 points)*

### Part A: Indirect ownership
Let $M$ be the direct-ownership matrix, where $M_{ij}$ is the fraction of company $j$ held directly by company $i$. For companies A, B, C and D:

| Holder \ Company held | A | B | C | D |
| :--- | :--- | :--- | :--- | :--- |
| **A** | 0 | 0.6 | 0 | 0.1 |
| **B** | 0 | 0 | 0.5 | 0 |
| **C** | 0.2 | 0 | 0 | 0.4 |
| **D** | 0 | 0 | 0 | 0 |

Define total ownership (direct + indirect through every chain) as:
$$T = M + M^{2} + M^{3} + \dots$$

1. **(20%)** Prove that if the series converges, $T=(I-M)^{-1}-I$, and state the convergence condition in terms of the spectral radius. Explain why real-world ownership matrices usually satisfy this condition.

พิสูจน์ว่า
$$(I-M)^{-1}-I=M + M^{2} + M^{3} + \dots$$

Neumann series บอกว่าเมื่อ T < 1 
$$(I-T)^{-1}=I + T + T^{2} + T^{3} + \dots$$

ดังนั้นในสมการเมื่อ M < 1
$$(I-M)^{-1}=I + M + M^{2} + M^{3} + \dots$$
$$(I-M)^{-1}-I=I + M + M^{2} + M^{3} + \dots - I$$
$$(I-M)^{-1}-I=M + M^{2} + M^{3} + \dots$$

ในกรณีที่ M ≥ 1 จำทำให้ค่า $T = M + M^{2} + M^{3} + \dots$ เป็น infinity ซึ่งในสถานการณืของการถือหุ้น ค่อนข้างมีน้อยที่บริษัทอื่นภายนอกจะมีการถือหุ้นแบบ 100% (M = 1) เนื่องจากเจ้าของบริษัทเองมักจะมีการถือหุ้นบริษัทตัวเองไว้อยู่เสมอ

2. **(27%)** Compute $T$ for the table above, both by hand (show how the structure of the matrix simplifies the calculation) and with NumPy. What is A's total percentage ownership of D?

3. **(13%)** The diagonal entries of $T$ for A, B and C are non-zero. What does this mean in business terms, and how should it be reported to business stakeholders?

หมายความว่าบริษัทมีการถือหุ้นของบริษัทตัวเองในทางอ้อมอยู่ทำให้เกิด circular ownership ทำให้เงินทุนรวมอาจจะสูงเกินจริงในทางบัญชีจึงต้องมีการชี้แจ้งแล้วหักค่านี้ออกเมื่อคำนวณมูลค่าจริงของบริษัทจากบุคคลภายนอก

### Part B: PCA and SVD
Let $X$ be a centred $n \times p$ data matrix.

1. **(20%)** Prove that the unit vector $w$ that maximises the variance of $Xw$ is the eigenvector of the covariance matrix with the largest eigenvalue (use a Lagrange multiplier).

2. **(13%)** Show the relationship between the principal components and the SVD $X=U\Sigma V^{T}$, and explain why in practice PCA should be computed via the SVD rather than by an eigen-decomposition of $X^{T}X$.

3. **(7%)** When must variables be standardised before PCA? Give an example using company financial statement data

## Problem 3 - Probability & Statistics: Data-driven Decisions *(elective, 50 points)*

### Part A: Bayes and fraud detection
A fraud-detection system has 95% sensitivity and 98% specificity, while only 1% of transactions are actually fraudulent.

1. **(10%)** When the system raises an alert, what is the probability that the transaction is truly fraudulent?

**ประมาณ 1.03%**
โดยคำนวณจาก Bayes Theorem $$P(Fraud|Flagged) = \frac{P(Flagged|Fraud)*P(Fraud)}{P(Flagged)}$$ จะได้ว่าความเป็นไปได้ที่ transaction จะ fraudulent จริง ๆ อยู่ที่ $\frac{0.01*0.98}{0.95}=0.0103$ หรือ 1.03%

2. **(10%)** With 1,000,000 transactions per month and an investigation team that can review 10,000 transactions per month, is this system workable in practice? Support your answer with numbers.

3. **(10%)** Propose at least two ways to improve the process (e.g. cost-based threshold selection, or multi-stage review) and quantify the impact of at least one.

### Part B: A/B testing
A team tests a new sign-up page with 5,000 users per group. The control group converts at 4.0% and the treatment group at 4.6%.

1. **(15%)** Test the hypothesis at $\alpha=0.05$. Report the test statistic, the p-value and a 95% confidence interval for the difference.

จากผลการวิเคราะห์ข้อมูล พบว่า treatment group ไม่มีผลต่อ control group อย่างมีนัยสำคัญทางสถิติที่ระดับ p < 0.05 (z = 1.4789, p = 0.1392) จึงปฏิเสธสมมติฐานที่ในการเปลี่ยนแปลง sign-up page

ซึ่งคำนวณได้จากการคำนวณหา z-score ของโจทย์ จากสูตรนี้ (คำนวณที่ ./src/q3/3b1_hypothesis.py)
$$\hat{p} = \frac{x_1 + x_2}{n_1 + n_2}$$
$$SE_{\text{pooled}} = \sqrt{\hat{p}(1 - \hat{p})\left(\frac{1}{n_1} + \frac{1}{n_2}\right)}$$
$$z = \frac{x_1 - x_2}{SE_{\text{pooled}}}$$

2. **(15%)** Find the per-group sample size needed to detect a change from 4.0% to 4.6% with 80% power, and explain what the result of question 1 tells you about the design of this experiment.

3. **(10%)** A manager checks the results every day and stops the experiment as soon as $p<0.05$. Write a simulation (A/A tests, at least 1,000 runs) showing what the actual false-positive rate is.

4. **(10%)** The team measured 20 metrics at once and found one with $p=0.03$. What should they conclude? Explain using Bonferroni and Benjamini-Hochberg.

### Part C: Conceptual understanding
**(20%)** Explain Simpson's paradox with a numerical example of your own (e.g. loan-approval rates broken down by business size), and describe how you would decide whether to look at the aggregated result or the per-group results.